# Reconnaissance d'entités nommées

In [ ]:
# Cellule 1 — Installation
!pip install lxml pandas spacy
!python -m spacy download fr_core_news_md

In [ ]:
from pathlib import Path
from lxml import etree
import pandas as pd
import spacy
import json

DATA_DIR = Path("../data")

TEI_IN = DATA_DIR / "Frêne_volume_1.xml"

TEI_OUT = DATA_DIR / "Frêne_volume_1_ner.xml"

JSON_LINES = DATA_DIR / "Frêne_volume_1_lignes.json"

JSON_ENRICHED = DATA_DIR / "Frêne_volume_1_lignes_ner.json"

ENTITIES_CSV = DATA_DIR / "Frêne_volume_1_entities.csv"

NS = {
    "tei": "http://www.tei-c.org/ns/1.0",
    "xml": "http://www.w3.org/XML/1998/namespace",
}

TEI_NS = "http://www.tei-c.org/ns/1.0"

nlp = spacy.load("fr_core_news_md")

In [ ]:
SPACY_TO_TEI = {
    "PER": "persName",
    "PERSON": "persName",
    "LOC": "placeName",
    "GPE": "placeName",
    "ORG": "orgName",
    "MISC": "name",
    "DATE": "date",
}

In [ ]:
def extraire_lignes_tei_vers_json(tei_path, json_out):
    """
    Extrait les lignes d'un fichier TEI et les sauvegarde dans un JSON structuré.

    Chaque ligne contient :
    - son xml:id ;
    - la surface/page à laquelle elle appartient ;
    - son numéro de ligne ;
    - son texte brut ;
    - une liste vide d'entités, qui sera remplie ensuite.
    """

    tree = etree.parse(str(tei_path))
    root = tree.getroot()

    lignes = []

    for line in root.findall(".//tei:line", namespaces=NS):
        xml_id = line.get("{http://www.w3.org/XML/1998/namespace}id")
        line_n = line.get("n")
        text = "".join(line.itertext()).strip()

        if not text:
            continue

        surface = line.getparent()

        while surface is not None and etree.QName(surface).localname != "surface":
            surface = surface.getparent()

        surface_id = None
        surface_n = None

        if surface is not None:
            surface_id = surface.get("{http://www.w3.org/XML/1998/namespace}id")
            surface_n = surface.get("n")

        lignes.append({
            "xml_id": xml_id,
            "surface_id": surface_id,
            "surface_n": surface_n,
            "line_n": line_n,
            "text": text,
            "entities": []
        })

    with open(json_out, "w", encoding="utf-8") as f:
        json.dump(lignes, f, ensure_ascii=False, indent=2)

    print(f"JSON des lignes créé : {json_out}")

    return lignes


lignes = extraire_lignes_tei_vers_json(TEI_IN, JSON_LINES)

In [ ]:
def enrichir_json_avec_ner(json_in, json_out, csv_out=None):
    """
    Lit un JSON de lignes TEI, applique spaCy sur chaque ligne,
    puis ajoute les entités reconnues directement dans la structure JSON.
    """

    with open(json_in, "r", encoding="utf-8") as f:
        lignes = json.load(f)

    toutes_entites = []

    for ligne in lignes:
        doc = nlp(ligne["text"])

        entites = []

        for ent in doc.ents:
            tei_tag = SPACY_TO_TEI.get(ent.label_, "name")

            entite = {
                "text": ent.text,
                "label_spacy": ent.label_,
                "tei_tag": tei_tag,
                "start_char": ent.start_char,
                "end_char": ent.end_char
            }

            entites.append(entite)

            toutes_entites.append({
                "xml_id": ligne["xml_id"],
                "surface_id": ligne["surface_id"],
                "surface_n": ligne["surface_n"],
                "line_n": ligne["line_n"],
                "entity_text": ent.text,
                "label_spacy": ent.label_,
                "tei_tag": tei_tag,
                "start_char": ent.start_char,
                "end_char": ent.end_char
            })

        ligne["entities"] = entites

    with open(json_out, "w", encoding="utf-8") as f:
        json.dump(lignes, f, ensure_ascii=False, indent=2)

    print(f"JSON enrichi créé : {json_out}")

    if csv_out is not None:
        df = pd.DataFrame(toutes_entites)
        df.to_csv(csv_out, index=False, encoding="utf-8")
        print(f"CSV des entités créé : {csv_out}")

    return lignes


lignes_enrichies = enrichir_json_avec_ner(
    JSON_LINES,
    JSON_ENRICHED,
    ENTITIES_CSV
)

In [ ]:
def construire_ligne_annotee_depuis_json(text, entities):
    """
    Reconstruit le contenu XML d'une ligne TEI à partir du texte brut
    et des entités stockées dans le JSON.
    """

    parent = etree.Element("tmp")
    pos = 0
    last_node = None

    entities = sorted(entities, key=lambda e: e["start_char"])

    for ent in entities:
        start = int(ent["start_char"])
        end = int(ent["end_char"])

        # Ignore les entités qui se chevauchent avec une entité déjà traitée.
        if start < pos:
            continue

        before = text[pos:start]
        ent_text = text[start:end]

        tei_tag = ent.get("tei_tag", "name")

        if last_node is None:
            parent.text = (parent.text or "") + before
        else:
            last_node.tail = (last_node.tail or "") + before

        element = etree.Element(f"{{{TEI_NS}}}{tei_tag}")
        element.text = ent_text
        element.set("type", ent.get("label_spacy", ""))

        parent.append(element)
        last_node = element

        pos = end

    after = text[pos:]

    if last_node is None:
        parent.text = (parent.text or "") + after
    else:
        last_node.tail = (last_node.tail or "") + after

    return parent

In [ ]:
def reinjecter_json_dans_tei(tei_in, json_enriched, tei_out):
    """
    Réinjecte les entités stockées dans le JSON enrichi
    dans une copie du fichier TEI original.
    """

    with open(json_enriched, "r", encoding="utf-8") as f:
        lignes = json.load(f)

    lignes_par_id = {
        ligne["xml_id"]: ligne
        for ligne in lignes
        if ligne.get("xml_id")
    }

    tree = etree.parse(str(tei_in))
    root = tree.getroot()

    for line in root.findall(".//tei:line", namespaces=NS):
        xml_id = line.get("{http://www.w3.org/XML/1998/namespace}id")

        if xml_id not in lignes_par_id:
            continue

        ligne_json = lignes_par_id[xml_id]
        entities = ligne_json.get("entities", [])

        if not entities:
            continue

        original_text = ligne_json["text"]

        annotated = construire_ligne_annotee_depuis_json(
            original_text,
            entities
        )

        anciens_attributs = dict(line.attrib)

        line.clear(keep_tail=True)

        for key, value in anciens_attributs.items():
            line.set(key, value)

        line.text = annotated.text

        for child in annotated:
            line.append(child)

    tree.write(
        str(tei_out),
        encoding="UTF-8",
        xml_declaration=True,
        pretty_print=True
    )

    print(f"TEI enrichi créé : {tei_out}")


reinjecter_json_dans_tei(
    TEI_IN,
    JSON_ENRICHED,
    TEI_OUT
)

In [ ]:
tree = etree.parse(str(TEI_OUT))

print("persName :", len(tree.findall(".//tei:persName", namespaces=NS)))
print("placeName :", len(tree.findall(".//tei:placeName", namespaces=NS)))
print("orgName :", len(tree.findall(".//tei:orgName", namespaces=NS)))
print("date :", len(tree.findall(".//tei:date", namespaces=NS)))
print("name :", len(tree.findall(".//tei:name", namespaces=NS)))